In [2]:
import pandas as pd
import numpy as np
import boto3
import io
import json
import datetime
import warnings
warnings.filterwarnings('ignore')

BUCKET_NAME = "churn-mlops-project-cherry"
s3 = boto3.client('s3')

def load_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

# Reference = training data (what model learned from)
# Current   = new incoming data (what model sees now)
reference_data = load_from_s3(BUCKET_NAME, 'data/features/X_train.csv')
current_data   = load_from_s3(BUCKET_NAME, 'data/features/X_test.csv')

print("✅ Data loaded!")
print(f"📊 Reference data shape: {reference_data.shape}")
print(f"📊 Current data shape:   {current_data.shape}")

✅ Data loaded!
📊 Reference data shape: (911, 32)
📊 Current data shape:   (228, 32)


In [3]:
# PSI = Population Stability Index
# Used by Netflix, banks, and top tech companies
# to detect when data has shifted
# PSI < 0.1  = stable, no action needed
# PSI 0.1-0.2 = monitor closely
# PSI > 0.2  = retrain model immediately!

def calculate_psi(expected, actual, buckets=10):
    expected = np.array(expected)
    actual   = np.array(actual)

    breakpoints = np.percentile(expected,
                                np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints)

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts   = np.histogram(actual,   bins=breakpoints)[0]

    expected_pct = expected_counts / len(expected)
    actual_pct   = actual_counts   / len(actual)

    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct   = np.where(actual_pct   == 0, 0.0001, actual_pct)

    psi = np.sum((actual_pct - expected_pct) *
                  np.log(actual_pct / expected_pct))
    return psi

# Features to monitor
features_to_monitor = [
    'engagement_score',
    'watch_hours',
    'last_login_days',
    'monthly_fee',
    'avg_watch_time_per_day',
    'fee_per_profile',
    'watch_to_fee_ratio'
]

print("📊 PSI DRIFT SCORES:")
print("="*55)
print(f"{'Feature':<35} {'PSI':>8}  {'Status'}")
print("-"*55)

psi_results = {}
for feature in features_to_monitor:
    if feature in reference_data.columns and feature in current_data.columns:
        psi = calculate_psi(
            reference_data[feature].dropna(),
            current_data[feature].dropna()
        )
        psi_results[feature] = psi

        if psi < 0.1:
            status = "✅ Stable"
        elif psi < 0.2:
            status = "⚠️  Monitor"
        else:
            status = "🚨 Retrain!"

        print(f"{feature:<35} {psi:>8.4f}  {status}")

print()
max_psi = max(psi_results.values())
print(f"Highest PSI: {max_psi:.4f}")
if max_psi > 0.2:
    print("🚨 ACTION NEEDED: Retrain the model!")
elif max_psi > 0.1:
    print("⚠️  MONITOR: Keep watching these features")
else:
    print("✅ ALL GOOD: Model is stable")

📊 PSI DRIFT SCORES:
Feature                                  PSI  Status
-------------------------------------------------------
engagement_score                      0.0530  ✅ Stable
watch_hours                           0.0345  ✅ Stable
last_login_days                       0.0335  ✅ Stable
monthly_fee                           0.0092  ✅ Stable
avg_watch_time_per_day                0.0632  ✅ Stable
fee_per_profile                       0.0327  ✅ Stable
watch_to_fee_ratio                    0.0374  ✅ Stable

Highest PSI: 0.0632
✅ ALL GOOD: Model is stable


In [4]:
# KS Test = Kolmogorov-Smirnov Test
# This is a statistical test that checks if two
# datasets come from the same distribution
# p-value < 0.05 means drift is detected!

from scipy import stats

print("📊 KS TEST RESULTS:")
print("="*60)
print(f"{'Feature':<35} {'p-value':>10}  {'Status'}")
print("-"*60)

ks_results = {}
for feature in features_to_monitor:
    if feature in reference_data.columns and feature in current_data.columns:
        ref = reference_data[feature].dropna()
        cur = current_data[feature].dropna()

        ks_stat, p_value = stats.ks_2samp(ref, cur)
        ks_results[feature] = p_value

        if p_value < 0.05:
            status = "🚨 Drift detected"
        else:
            status = "✅ No drift"

        print(f"{feature:<35} {p_value:>10.4f}  {status}")

drifted = sum(1 for p in ks_results.values() if p < 0.05)
print(f"\nDrifted features: {drifted} out of {len(ks_results)}")

📊 KS TEST RESULTS:
Feature                                p-value  Status
------------------------------------------------------------
engagement_score                        0.4632  ✅ No drift
watch_hours                             0.5979  ✅ No drift
last_login_days                         0.5601  ✅ No drift
monthly_fee                             0.8405  ✅ No drift
avg_watch_time_per_day                  0.5244  ✅ No drift
fee_per_profile                         0.8683  ✅ No drift
watch_to_fee_ratio                      0.7908  ✅ No drift

Drifted features: 0 out of 7


In [5]:
# Check predictions stored in S3
# Are we predicting more churners than usual?
# A sudden spike in churn predictions = something changed!

import joblib

# Load model and scaler
s3.download_file(BUCKET_NAME, 'models/latest/xgboost_churn_model.pkl', '/tmp/model.pkl')
s3.download_file(BUCKET_NAME, 'models/scaler.pkl', '/tmp/scaler.pkl')
model  = joblib.load('/tmp/model.pkl')
scaler = joblib.load('/tmp/scaler.pkl')

# Score current data
current_scaled = scaler.transform(current_data)
predictions    = model.predict_proba(current_scaled)[:, 1]

# Score reference data
reference_scaled = scaler.transform(reference_data)
ref_predictions  = model.predict_proba(reference_scaled)[:, 1]

print("📊 PREDICTION MONITORING:")
print("="*50)
print(f"Reference data predictions:")
print(f"   Average churn probability: {ref_predictions.mean():.3f}")
print(f"   Predicted churn rate:      {(ref_predictions >= 0.5).mean():.1%}")
print()
print(f"Current data predictions:")
print(f"   Average churn probability: {predictions.mean():.3f}")
print(f"   Predicted churn rate:      {(predictions >= 0.5).mean():.1%}")
print()

# Check if prediction distribution shifted
prob_shift = abs(predictions.mean() - ref_predictions.mean())
print(f"Prediction shift: {prob_shift:.4f}")
if prob_shift > 0.1:
    print("🚨 ALERT: Prediction distribution has shifted significantly!")
    print("   Model may be giving unreliable predictions")
else:
    print("✅ Predictions are stable")

📊 PREDICTION MONITORING:
Reference data predictions:
   Average churn probability: 0.965
   Predicted churn rate:      98.2%

Current data predictions:
   Average churn probability: 0.951
   Predicted churn rate:      96.1%

Prediction shift: 0.0135
✅ Predictions are stable


In [6]:
# Save everything as a monitoring report
# This is what a real MLOps dashboard would read

monitoring_report = {
    "checked_at":     str(datetime.datetime.now()),
    "psi_scores":     {k: round(float(v), 4) for k, v in psi_results.items()},
    "max_psi":        round(float(max(psi_results.values())), 4),
    "ks_p_values":    {k: round(float(v), 4) for k, v in ks_results.items()},
    "drifted_features_ks": int(drifted),
    "avg_churn_prob_reference": round(float(ref_predictions.mean()), 4),
    "avg_churn_prob_current":   round(float(predictions.mean()), 4),
    "prediction_shift":         round(float(prob_shift), 4),
    "action_needed": bool(
        max(psi_results.values()) > 0.2 or
        drifted > 2 or
        prob_shift > 0.1
    ),
    "recommendation": (
        "Retrain model immediately" if max(psi_results.values()) > 0.2 else
        "Monitor closely"           if max(psi_results.values()) > 0.1 else
        "Model is stable"
    )
}

# Save to S3
s3.put_object(
    Bucket=BUCKET_NAME,
    Key='monitoring/latest_summary.json',
    Body=json.dumps(monitoring_report, indent=2)
)

print("✅ Monitoring report saved to S3!")
print(f"\n📊 Full Report:")
print(json.dumps(monitoring_report, indent=2))
print(f"\n🎉 Step 6 Complete!")
print(f"Your model is now being monitored!")
print(f"PSI drift detection ✅")
print(f"KS statistical test ✅")
print(f"Prediction monitoring ✅")
print(f"Report saved to S3 ✅")

✅ Monitoring report saved to S3!

📊 Full Report:
{
  "checked_at": "2026-04-14 06:46:29.768992",
  "psi_scores": {
    "engagement_score": 0.053,
    "watch_hours": 0.0345,
    "last_login_days": 0.0335,
    "monthly_fee": 0.0092,
    "avg_watch_time_per_day": 0.0632,
    "fee_per_profile": 0.0327,
    "watch_to_fee_ratio": 0.0374
  },
  "max_psi": 0.0632,
  "ks_p_values": {
    "engagement_score": 0.4632,
    "watch_hours": 0.5979,
    "last_login_days": 0.5601,
    "monthly_fee": 0.8405,
    "avg_watch_time_per_day": 0.5244,
    "fee_per_profile": 0.8683,
    "watch_to_fee_ratio": 0.7908
  },
  "drifted_features_ks": 0,
  "avg_churn_prob_reference": 0.9649,
  "avg_churn_prob_current": 0.9515,
  "prediction_shift": 0.0135,
  "action_needed": false,
  "recommendation": "Model is stable"
}

🎉 Step 6 Complete!
Your model is now being monitored!
PSI drift detection ✅
KS statistical test ✅
Prediction monitoring ✅
Report saved to S3 ✅
